In [ ]:
import os
from collections import namedtuple
from onekey_algo.classification3d.run_classification3d import main as clf_main3d
from onekey_algo import get_param_in_cwd

# 设置参数
roi_size = [64, 64, 48]
vit_settings = {
    'image_size': roi_size[0],
    'frames': roi_size[-1],
    'image_patch_size': 16,
    'frame_patch_size': 2,
    'dim': 1024,
    'depth': 6,
    'heads': 8,
    'mlp_dim': 2048,
    'dropout': 0.1,
    'emb_dropout': 0.1
}

# 获取文件路径
train_file = get_param_in_cwd('train_file')
val_file = get_param_in_cwd('val_file')
data_pattern = get_param_in_cwd('data_pattern')
labels_file = r'labels.txt'

# 1. 读取训练文件，获取所有唯一的标签
def extract_labels_from_file(file_path):
    labels = set()
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    parts = line.split()
                    if len(parts) >= 2:
                        label = parts[1]  # 第二个元素是标签
                        labels.add(label)
    return labels

# 从训练文件和验证文件中提取标签
train_labels = extract_labels_from_file(train_file)
val_labels = extract_labels_from_file(val_file)
all_labels = train_labels.union(val_labels)

print(f"训练文件中的标签: {train_labels}")
print(f"验证文件中的标签: {val_labels}")
print(f"所有标签: {all_labels}")

# 2. 创建labels.txt文件 - 只包含标签值，每行一个
with open(labels_file, 'w', encoding='utf-8') as f:
    for label in sorted(all_labels):
        f.write(f"{label}\n")

print(f"\n创建的 labels.txt 内容:")
with open(labels_file, 'r', encoding='utf-8') as f:
    content = f.read()
    print(content)
    print(f"行数: {len(content.strip().splitlines())}")

# 3. 验证文件内容是否匹配
print("\n验证标签匹配:")
print(f"从labels.txt读取的标签: {set(content.strip().splitlines())}")
print(f"从训练文件读取的标签: {all_labels}")
print(f"是否匹配: {set(content.strip().splitlines()) == all_labels}")

# 4. 确保data_pattern正确
if not data_pattern or data_pattern == '':
    data_dir = os.path.dirname(train_file)
    data_pattern = os.path.join(data_dir, '{}')
    print(f"\n设置data_pattern为: {data_pattern}")

# 5. 添加额外的调试信息
print(f"\n文件路径:")
print(f"训练文件: {train_file}")
print(f"验证文件: {val_file}")
print(f"标签文件: {labels_file}")
print(f"数据模式: {data_pattern}")

# 6. 检查训练文件的前几行
print(f"\n训练文件前5行:")
with open(train_file, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(f"  {i+1}: {line.strip()}")

# 7. 指定要训练的模型列表
models_to_train = ['ResNet101', 'DenseNet121', 'SimpleViT']

# 8. 开始训练 - 只训练指定的三个模型
for model_name in models_to_train:
    print(f"\n{'='*60}")
    print(f"开始训练模型: {model_name}")
    print(f"{'='*60}")
    
    params = dict(
        train=train_file,
        val=val_file,
        labels_file=labels_file,
        data_pattern=data_pattern,
        j=0,
        max2use=None,
        normalize_method='imagenet',
        model_name=model_name,
        gpus=[0],
        roi_size=roi_size,
        batch_size=4,
        epochs=get_param_in_cwd('epoch'),
        init_lr=0.001,
        optimizer='adam',
        retrain=None,
        model_root=get_param_in_cwd('model_root'),
        val_interval=1,
        save_per_epoch=False,
        iters_verbose=10,
        model_config={'groups': 1, 'blocks_args': 'efficientnet-b4'},
        vit_settings=vit_settings,
        pretrained=True,
        dataset_name='list'
    )
    
    try:
        print(f"\n开始训练...")
        Args = namedtuple("Args", params.keys())
        args_instance = Args(**params)
        clf_main3d(args_instance)
        print(f"模型 {model_name} 训练完成!")
        
    except Exception as e:
        print(f"\n训练模型 {model_name} 时出错: {e}")
        import traceback
        traceback.print_exc()
        print(f"跳过模型 {model_name}")
        continue

print("\n所有指定模型训练完成!")

In [ ]:
#深度学习特征提取
import os
from onekey_algo import OnekeyDS as okds
from onekey_algo import get_param_in_cwd
# 目录模式
mydir = get_param_in_cwd('data_pattern')
directory = os.path.expanduser(mydir)
test_samples = [os.path.join(directory, p) 
                for p in os.listdir(directory) if p.endswith('.nii') or p.endswith('.nii.gz')]
test_samples


#读取数据
import pandas as pd
features = pd.read_csv(f'features/{model_name}_features.csv', header=None)
features.columns=['ID'] + [f"DL_{i}" for i in range(features.shape[1] - 1)]
features.to_csv(f'features/{model_name}_features.csv', index=False)
features


#深度学习特征压缩
from onekey_algo.custom.components.comp1 import compress_df_feature

cm_features = compress_df_feature(features=features, dim=32, prefix='DL_', not_compress='ID')
cm_features.to_csv(f'features/{model_name}_compress_features.csv', header=True, index=False)


import pandas as pd
import os

# 直接指定文件路径和前缀
input_file = r""
prefix = ""

# 读取文件
df = pd.read_csv(input_file)

# 为除第一列外的所有列添加前缀
columns = df.columns.tolist()
new_columns = [columns[0]] + [prefix + col for col in columns[1:]]
df.columns = new_columns

# 保存文件
output_file = os.path.join(os.path.dirname(input_file), f"{prefix}{os.path.basename(input_file)}")
df.to_csv(output_file, index=False)

print(f"处理完成！文件已保存为: {output_file}")
print("处理后的列名（前5列）:")
for col in df.columns[:5]:
    print(f"  {col}")